In [ ]:
"""
===========================================================================
  FILE:    main.ipynb
  TITLE:   Maternal Education and the Demand for Maternal & Child Health
           Services in Bangladesh -- BDHS 2022
  DATA:    BDNR81FL.DTA  (Pregnancy & Postnatal Care Recode, NR file)
  DATE:    March 2026

  REQUIREMENTS:
      pip install pandas numpy scipy matplotlib seaborn statsmodels openpyxl
===========================================================================
"""

In [4]:
# ---------------------------------------------------------------------------
# SECTION 0 -- IMPORTS & PATHS
# ---------------------------------------------------------------------------
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Probit, Logit

warnings.filterwarnings("ignore")

# ── Change these two paths before running ───────────────────────────────────
DATA_DIR = r"E:\7th semester Econometrics\data\Pregnancy and Postnatal Care Recode.DTA"
OUT_DIR  = r"E:\7th semester Econometrics\data\output"
NR_FILE  = pd.read_stata(DATA_DIR)

os.makedirs(OUT_DIR, exist_ok=True)
print(f"Output directory: {OUT_DIR}")

Output directory: E:\7th semester Econometrics\data\output


In [7]:
# ---------------------------------------------------------------------------
# SECTION 1 -- LOAD ONLY NEEDED COLUMNS
# ---------------------------------------------------------------------------
print("\n[1] Loading data ...")

KEEP_COLS = [
    "m14",   # number of ANC visits
    "m15",   # place of delivery
    "m2a",   # prenatal care by doctor (1=yes, 0=no)
    "m45",   # iron supplementation (1=yes, 0=no)
    "m13",   # timing of first ANC in months
    "v106",  # education level (0=none,1=primary,2=secondary,3=higher)
    "v133",  # years of schooling
    "v025",  # residence (1=urban, 2=rural)
    "v190",  # wealth quintile (1=poorest...5=richest)
    "v012",  # mother's current age
    "v024",  # division (1=Barisal...8=Sylhet)
    "pidx",  # birth/parity order
    "v005",  # sample weight (divide by 1,000,000)
]

df_raw = pd.read_stata(DATA_DIR, convert_categoricals=False, columns=KEEP_COLS)
print(f"    Rows: {len(df_raw):,}   Columns kept: {df_raw.shape[1]}")


[1] Loading data ...
    Rows: 6,315   Columns kept: 13


In [8]:
# ---------------------------------------------------------------------------
# SECTION 2 -- CONSTRUCT OUTCOME VARIABLES
# ---------------------------------------------------------------------------

print("\n[2] Constructing outcome variables ...")

df = df_raw.copy()

# (1) ANC 4+ visits
df["anc4"] = np.where(
    df["m14"].isna() | (df["m14"] >= 90), np.nan,
    (df["m14"] >= 4).astype(float)
)

# (2) Facility-based delivery
FACILITY_CODES = set(range(20, 30)) | set(range(31, 47))
df["facility_del"] = np.where(
    df["m15"].isna() | df["m15"].isin([98, 99]), np.nan,
    df["m15"].isin(FACILITY_CODES).astype(float)
)

# (3) Doctor-attended ANC
df["anc_doctor"] = np.where(
    df["m2a"].isin([0, 1]), df["m2a"].astype(float), np.nan
)

# (4) Iron supplementation
df["iron_supp"] = np.where(
    df["m45"].isin([0, 1]), df["m45"].astype(float), np.nan
)

# (5) First ANC in first trimester (<= 3 months)
df["anc_first"] = np.where(
    df["m13"].isna() | (df["m13"] >= 90), np.nan,
    (df["m13"] <= 3).astype(float)
)

OUTCOMES = {
    "anc4":        "ANC 4+ Visits",
    "facility_del":"Facility Delivery",
    "anc_doctor":  "Doctor-Attended ANC",
    "iron_supp":   "Iron Supplementation",
    "anc_first":   "1st ANC in 1st Trimester",
}

print("\n    Outcome prevalences:")
for col, lbl in OUTCOMES.items():
    n   = df[col].notna().sum()
    pct = df[col].mean() * 100
    print(f"      {lbl:<30s}  N={n:,}   {pct:.1f}%")


[2] Constructing outcome variables ...

    Outcome prevalences:
      ANC 4+ Visits                   N=5,186   40.5%
      Facility Delivery               N=5,469   63.7%
      Doctor-Attended ANC             N=5,187   83.3%
      Iron Supplementation            N=5,185   79.0%
      1st ANC in 1st Trimester        N=4,758   42.2%


In [30]:
# ---------------------------------------------------------------------------
# SECTION 3 -- CONSTRUCT EXPLANATORY & CONTROL VARIABLES
# ---------------------------------------------------------------------------
print("\n[3] Constructing predictors ...")

# Education dummies (ref = no education = 0)
df["edu_level"]     = df["v106"]
df["edu_primary"]   = (df["v106"] == 1).astype(float)
df["edu_secondary"] = (df["v106"] == 2).astype(float)
df["edu_higher"]    = (df["v106"] == 3).astype(float)
df["edu_yrs"]       = df["v133"].where(df["v133"] <= 20)

# Wealth dummies (ref = poorest = 1)
df["wealth"] = df["v190"]
for q in [2, 3, 4, 5]:
    df[f"wealth_q{q}"] = (df["v190"] == q).astype(float)

# Residence
df["rural"] = df["rural"].where(df["v025"].notna(), other=np.nan)

# Mother's age (raw + standardized)
df["mother_age"]     = df["v012"]
df["mother_age_std"] = (df["v012"] - df["v012"].mean()) / df["v012"].std()

# Parity dummies (ref = first birth)
df["par_2"]     = (df["pidx"] == 2).astype(float)
df["par_3plus"] = (df["pidx"] >= 3).astype(float)

# Division dummies (ref = Barisal = 1)
DIV_LABELS = {
    1:"Barisal", 2:"Chattogram", 3:"Dhaka",    4:"Khulna",
    5:"Mymensingh", 6:"Rajshahi", 7:"Rangpur", 8:"Sylhet",
}
DIVISION_DUMMIES = []
for code, name in DIV_LABELS.items():
    if code != 1:
        col_name = f"div_{name}"
        df[col_name] = (df["v024"] == code).astype(float)
        DIVISION_DUMMIES.append(col_name)

# Survey weight
df["weight"] = df["v005"] / 1_000_000

# Master regressor list
REGRESSORS = (
    ["edu_primary", "edu_secondary", "edu_higher",
     "wealth_q2", "wealth_q3", "wealth_q4", "wealth_q5",
     "rural", "mother_age_std", "par_2", "par_3plus"]
    + DIVISION_DUMMIES
)

VAR_NAMES = ["const"] + REGRESSORS   # used when indexing param arrays

REG_LABELS = {
    "const":          "Constant",
    "edu_primary":    "Primary (ref: no edu)",
    "edu_secondary":  "Secondary (ref: no edu)",
    "edu_higher":     "Higher (ref: no edu)",
    "wealth_q2":      "Poor Q2 (ref: poorest)",
    "wealth_q3":      "Middle Q3",
    "wealth_q4":      "Rich Q4",
    "wealth_q5":      "Richest Q5",
    "rural":          "Rural (ref: urban)",
    "mother_age_std": "Mother Age (standardised)",
    "par_2":          "Parity: 2nd",
    "par_3plus":      "Parity: 3rd+",
    "div_Chattogram": "Division: Chattogram",
    "div_Dhaka":      "Division: Dhaka",
    "div_Khulna":     "Division: Khulna",
    "div_Mymensingh": "Division: Mymensingh",
    "div_Rajshahi":   "Division: Rajshahi",
    "div_Rangpur":    "Division: Rangpur",
    "div_Sylhet":     "Division: Sylhet",
}

# Drop rows missing on any regressor (outcomes are allowed to differ)
df_clean = df[list(OUTCOMES.keys()) + REGRESSORS + ["edu_level","wealth","v024","rural"]].copy()


[3] Constructing predictors ...


In [10]:
# ---------------------------------------------------------------------------
# SECTION 4 -- DESCRIPTIVE STATISTICS  (Table 1)
# ---------------------------------------------------------------------------
print("\n[4] Descriptive statistics ...")

DESC_VARS = list(OUTCOMES.keys()) + [
    "edu_primary", "edu_secondary", "edu_higher",
    "wealth", "rural", "mother_age", "par_2", "par_3plus"
]
DESC_LABELS = {
    "anc4":          "ANC 4+ Visits",
    "facility_del":  "Facility Delivery",
    "anc_doctor":    "Doctor-Attended ANC",
    "iron_supp":     "Iron Supplementation",
    "anc_first":     "1st ANC in 1st Trimester",
    "edu_primary":   "Education: Primary",
    "edu_secondary": "Education: Secondary",
    "edu_higher":    "Education: Higher",
    "wealth":        "Wealth Quintile (1=poorest, 5=richest)",
    "rural":         "Rural Residence",
    "mother_age":    "Mother's Age (years)",
    "par_2":         "Parity: 2nd birth",
    "par_3plus":     "Parity: 3rd+ birth",
}

rows = []
for v in DESC_VARS:
    s = df[v].dropna()
    rows.append({
        "Variable": DESC_LABELS.get(v, v),
        "N":        int(len(s)),
        "Mean":     round(s.mean(),  3),
        "Std Dev":  round(s.std(),   3),
        "Min":      round(s.min(),   3),
        "Max":      round(s.max(),   3),
    })

desc_df = pd.DataFrame(rows)
desc_df.to_excel(os.path.join(OUT_DIR, "table1_descriptives.xlsx"), index=False)
desc_df.to_csv( os.path.join(OUT_DIR,  "table1_descriptives.csv"),  index=False)
print("    Saved table1_descriptives.xlsx / .csv")
print(desc_df.to_string(index=False))


[4] Descriptive statistics ...
    Saved table1_descriptives.xlsx / .csv
                              Variable    N   Mean  Std Dev  Min  Max
                         ANC 4+ Visits 5186  0.405    0.491  0.0  1.0
                     Facility Delivery 5469  0.637    0.481  0.0  1.0
                   Doctor-Attended ANC 5187  0.833    0.373  0.0  1.0
                  Iron Supplementation 5185  0.790    0.407  0.0  1.0
              1st ANC in 1st Trimester 4758  0.422    0.494  0.0  1.0
                    Education: Primary 6315  0.233    0.423  0.0  1.0
                  Education: Secondary 6315  0.520    0.500  0.0  1.0
                     Education: Higher 6315  0.192    0.394  0.0  1.0
Wealth Quintile (1=poorest, 5=richest) 6315  2.972    1.420  1.0  5.0
                       Rural Residence 6315  0.662    0.473  0.0  1.0
                  Mother's Age (years) 6315 25.984    5.823 15.0 49.0
                     Parity: 2nd birth 6315  0.097    0.296  0.0  1.0
                

In [11]:
# ---------------------------------------------------------------------------
# SECTION 5 -- BIVARIATE TABLES BY EDUCATION & WEALTH
# ---------------------------------------------------------------------------
print("\n[5] Bivariate prevalence tables ...")

EDU_MAP    = {0:"No Education", 1:"Primary", 2:"Secondary", 3:"Higher"}
WEALTH_MAP = {1:"Poorest", 2:"Poor", 3:"Middle", 4:"Rich", 5:"Richest"}

edu_rows = []
for code, lbl in EDU_MAP.items():
    row = {"Education": lbl}
    for col, olbl in OUTCOMES.items():
        s = df[df["edu_level"] == code][col].dropna()
        row[olbl] = f"{s.mean()*100:.1f}% (N={len(s):,})"
    edu_rows.append(row)
edu_table = pd.DataFrame(edu_rows)
edu_table.to_excel(os.path.join(OUT_DIR, "tableA_by_education.xlsx"), index=False)
print("\n    By education:")
print(edu_table.to_string(index=False))

wealth_rows = []
for code, lbl in WEALTH_MAP.items():
    row = {"Wealth Quintile": lbl}
    for col, olbl in OUTCOMES.items():
        s = df[df["wealth"] == code][col].dropna()
        row[olbl] = f"{s.mean()*100:.1f}% (N={len(s):,})"
    wealth_rows.append(row)
wealth_table = pd.DataFrame(wealth_rows)
wealth_table.to_excel(os.path.join(OUT_DIR, "tableB_by_wealth.xlsx"), index=False)
print("\n    By wealth quintile:")
print(wealth_table.to_string(index=False))


[5] Bivariate prevalence tables ...

    By education:
   Education   ANC 4+ Visits Facility Delivery Doctor-Attended ANC Iron Supplementation 1st ANC in 1st Trimester
No Education   20.4% (N=275)     38.6% (N=293)       61.6% (N=276)        56.9% (N=276)            29.2% (N=202)
     Primary 26.8% (N=1,200)   46.2% (N=1,278)     74.6% (N=1,200)      69.4% (N=1,200)          30.1% (N=1,029)
   Secondary 40.0% (N=2,733)   65.6% (N=2,877)     85.2% (N=2,733)      81.3% (N=2,731)          40.2% (N=2,567)
      Higher   64.3% (N=978)   87.2% (N=1,021)       94.6% (N=978)        90.9% (N=978)            63.1% (N=960)

    By wealth quintile:
Wealth Quintile   ANC 4+ Visits Facility Delivery Doctor-Attended ANC Iron Supplementation 1st ANC in 1st Trimester
        Poorest 22.5% (N=1,087)   39.6% (N=1,162)     68.4% (N=1,087)      67.8% (N=1,087)            27.6% (N=888)
           Poor 30.1% (N=1,044)   54.3% (N=1,103)     78.8% (N=1,045)      75.3% (N=1,045)            31.0% (N=936)
      

In [12]:
# ---------------------------------------------------------------------------
# SECTION 6 -- FIGURES
# ---------------------------------------------------------------------------
print("\n[6] Generating figures ...")

EDU_LABELS_SHORT    = ["No\nEducation", "Primary", "Secondary", "Higher"]
WEALTH_LABELS_SHORT = ["Poorest", "Poor", "Middle", "Rich", "Richest"]
COLORS_EDU          = ["#d73027", "#fc8d59", "#91bfdb", "#4575b4"]

# ── Figure 1: Bar charts by education level ──────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
fig.suptitle(
    "Maternal Health Service Utilisation by Education Level\n"
    "Bangladesh DHS 2022",
    fontsize=13, fontweight="bold"
)
for idx, (col, title) in enumerate(OUTCOMES.items()):
    ax = axes[idx // 3][idx % 3]
    grp  = df.groupby("edu_level")[col].mean() * 100
    bars = ax.bar(EDU_LABELS_SHORT, grp.values,
                  color=COLORS_EDU, edgecolor="white", linewidth=0.7)
    ax.set_title(title, fontweight="bold", fontsize=10)
    ax.set_ylabel("Prevalence (%)", fontsize=9)
    ax.set_ylim(0, min(105, grp.max() * 1.25))
    for bar, val in zip(bars, grp.values):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 1,
                f"{val:.1f}%", ha="center", fontsize=8.5, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="x", labelsize=8)

# Last panel: ANC visits boxplot
ax = axes[1][2]
anc_bp = df[["m14", "edu_level"]].dropna()
anc_bp = anc_bp[anc_bp["m14"] < 20]
groups = [anc_bp[anc_bp["edu_level"] == e]["m14"].values for e in [0, 1, 2, 3]]
bp = ax.boxplot(groups, tick_labels=EDU_LABELS_SHORT, patch_artist=True,
                medianprops=dict(color="white", linewidth=2))
for patch, c in zip(bp["boxes"], COLORS_EDU):
    patch.set_facecolor(c)
ax.axhline(y=4, color="red", linestyle="--", linewidth=1.5,
           label="WHO minimum (4 visits)")
ax.set_title("ANC Visits Distribution", fontweight="bold", fontsize=10)
ax.set_ylabel("Number of ANC Visits", fontsize=9)
ax.legend(fontsize=8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "fig1_by_education.png"), dpi=160, bbox_inches="tight")
plt.close()
print("    Saved fig1_by_education.png")

# ── Figure 2: Gradient lines (education + wealth) ────────────────────────────
LINE_STYLES = [
    ("anc4",        "ANC 4+ Visits",      "#2166ac", "o-"),
    ("facility_del","Facility Delivery",  "#4dac26", "s-"),
    ("anc_doctor",  "Doctor-Attended ANC","#d01c8b", "^-"),
    ("iron_supp",   "Iron Supplementation","#f1a340","D-"),
]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for col, lbl, color, fmt in LINE_STYLES:
    grp = df.groupby("edu_level")[col].mean() * 100
    ax.plot([0, 1, 2, 3], grp.values, fmt, color=color,
            linewidth=2, markersize=7, label=lbl)
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(EDU_LABELS_SHORT, fontsize=10)
ax.set_xlabel("Mother's Education Level", fontsize=11)
ax.set_ylabel("Prevalence (%)", fontsize=11)
ax.set_title("By Maternal Education", fontweight="bold", fontsize=11)
ax.legend(fontsize=9, framealpha=0.9)
ax.set_ylim(0, 105)
ax.grid(axis="y", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1]
for col, lbl, color, fmt in LINE_STYLES:
    grp = df.groupby("wealth")[col].mean() * 100
    ax.plot([1, 2, 3, 4, 5], grp.values, fmt, color=color,
            linewidth=2, markersize=7, label=lbl)
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels(WEALTH_LABELS_SHORT, fontsize=10)
ax.set_xlabel("Household Wealth Quintile", fontsize=11)
ax.set_ylabel("Prevalence (%)", fontsize=11)
ax.set_title("By Wealth Quintile", fontweight="bold", fontsize=11)
ax.legend(fontsize=9, framealpha=0.9)
ax.set_ylim(0, 105)
ax.grid(axis="y", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.suptitle("Health Service Demand Gradients -- Bangladesh DHS 2022",
             fontsize=12, fontweight="bold")
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "fig2_gradients.png"), dpi=160, bbox_inches="tight")
plt.close()
print("    Saved fig2_gradients.png")


[6] Generating figures ...
    Saved fig1_by_education.png
    Saved fig2_gradients.png


In [18]:
# ---------------------------------------------------------------------------
# SECTION 7 -- PROBIT MODELS
# ---------------------------------------------------------------------------
print("\n[7] Running probit models ...")

def sig_stars(p):
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return ""

def fit_probit_ame(outcome_col, df_in, regressors):
    sub = df_in[[outcome_col] + regressors].dropna()
    y   = sub[outcome_col].values
    X   = sm.add_constant(sub[regressors].values)

    # Try newton first (analytical Hessian, most reliable covariance)
    # Fall back to nm if newton fails
    for method in ["newton", "bfgs", "nm"]:
        try:
            res = Probit(y, X).fit(
                disp=False, maxiter=300, method=method,
                warn_convergence=False
            )
            _ = res.cov_params()   # test covariance is available
            break
        except Exception:
            continue

    phi_mean = norm.pdf(X @ res.params).mean()
    ame      = phi_mean * res.params
    ame_se   = phi_mean * np.sqrt(np.diag(res.cov_params()))
    z        = ame / ame_se
    p        = 2 * (1 - norm.cdf(np.abs(z)))
    pr2      = 1 - res.llf / res.llnull
    return res, ame, ame_se, z, p, int(res.nobs), pr2

probit_results = {}
for col, lbl in OUTCOMES.items():
    print(f"  Probit: {lbl} ...", end=" ")
    try:
        res, ame, ame_se, z, p, n, pr2 = fit_probit_ame(col, df_clean, REGRESSORS)
        probit_results[col] = dict(
            result=res, ame=ame, ame_se=ame_se,
            z=z, pval=p, n=n, pseudo_r2=pr2, label=lbl
        )
        print(f"N={n:,}  PseudoR2={pr2:.4f}  Converged={res.mle_retvals['converged']}")
    except Exception as e:
        print(f"ERROR: {e}")

# Print results to console
print("\n    Probit average marginal effects:")
for col, r in probit_results.items():
    print(f"\n  {'='*65}")
    print(f"  OUTCOME: {r['label']}")
    print(f"  N={r['n']:,}   McFadden R2={r['pseudo_r2']:.4f}")
    print(f"  {'Variable':<32} {'AME':>8} {'SE':>8} {'z':>7} {'p':>7}")
    print(f"  {'-'*58}")
    for i, vname in enumerate(VAR_NAMES):
        print(f"  {REG_LABELS.get(vname,vname):<32} "
              f"{r['ame'][i]:>8.4f} {r['ame_se'][i]:>8.4f} "
              f"{r['z'][i]:>7.3f} {r['pval'][i]:>7.4f}  "
              f"{sig_stars(r['pval'][i])}")


[7] Running probit models ...
  Probit: ANC 4+ Visits ... N=5,186  PseudoR2=0.1084  Converged=True
  Probit: Facility Delivery ... N=5,469  PseudoR2=0.1419  Converged=True
  Probit: Doctor-Attended ANC ... N=5,187  PseudoR2=0.1010  Converged=True
  Probit: Iron Supplementation ... N=5,185  PseudoR2=0.0669  Converged=True
  Probit: 1st ANC in 1st Trimester ... N=4,758  PseudoR2=0.0851  Converged=True

    Probit average marginal effects:

  OUTCOME: ANC 4+ Visits
  N=5,186   McFadden R2=0.1084
  Variable                              AME       SE       z       p
  ----------------------------------------------------------
  Constant                          -0.3613   0.0390  -9.256  0.0000  ***
  Primary (ref: no edu)              0.0740   0.0333   2.225  0.0261  **
  Secondary (ref: no edu)            0.1720   0.0325   5.294  0.0000  ***
  Higher (ref: no edu)               0.3006   0.0353   8.515  0.0000  ***
  Poor Q2 (ref: poorest)             0.0534   0.0208   2.568  0.0102  **
  M

In [19]:
# ---------------------------------------------------------------------------
# SECTION 8 -- LOGIT MODELS (robustness)
# ---------------------------------------------------------------------------
print("\n[8] Running logit models (robustness) ...")

logit_results = {}
for col, lbl in OUTCOMES.items():
    print(f"  Logit: {lbl} ...", end=" ")
    try:
        sub = df_clean[[col] + REGRESSORS].dropna()
        y   = sub[col].values
        X   = sm.add_constant(sub[REGRESSORS].values)

        for method in ["newton", "bfgs", "nm"]:
            try:
                res = Logit(y, X).fit(
                    disp=False, maxiter=300, method=method,
                    warn_convergence=False
                )
                _ = res.cov_params()   # test covariance is available
                break
            except Exception:
                continue

        xb       = X @ res.params
        lam      = 1 / (1 + np.exp(-xb))
        dydx_avg = (lam * (1 - lam)).mean()
        ame      = dydx_avg * res.params
        ame_se   = dydx_avg * np.sqrt(np.diag(res.cov_params()))
        z        = ame / ame_se
        p        = 2 * (1 - norm.cdf(np.abs(z)))
        pr2      = 1 - res.llf / res.llnull
        logit_results[col] = dict(ame=ame, ame_se=ame_se, z=z, pval=p,
                                   n=int(res.nobs), pseudo_r2=pr2, label=lbl)
        print(f"N={int(res.nobs):,}  PseudoR2={pr2:.4f}")
    except Exception as e:
        print(f"ERROR: {e}")


[8] Running logit models (robustness) ...
  Logit: ANC 4+ Visits ... N=5,186  PseudoR2=0.1083
  Logit: Facility Delivery ... N=5,469  PseudoR2=0.1420
  Logit: Doctor-Attended ANC ... N=5,187  PseudoR2=0.1006
  Logit: Iron Supplementation ... N=5,185  PseudoR2=0.0671
  Logit: 1st ANC in 1st Trimester ... N=4,758  PseudoR2=0.0851


In [20]:
# ---------------------------------------------------------------------------
# SECTION 9 -- LPM / OLS (robustness)
# ---------------------------------------------------------------------------
print("\n[9] Running LPM (OLS) models (robustness) ...")

lpm_results = {}
for col, lbl in OUTCOMES.items():
    sub = df_clean[[col] + REGRESSORS].dropna()
    y   = sub[col].values
    X   = sm.add_constant(sub[REGRESSORS].values)
    res = sm.OLS(y, X).fit(cov_type="HC1")
    lpm_results[col] = dict(coef=res.params, se=res.bse,
                             t=res.tvalues, pval=res.pvalues,
                             n=int(res.nobs), r2=res.rsquared, label=lbl)
    print(f"  LPM {lbl:<30s}  N={int(res.nobs):,}  R2={res.rsquared:.4f}")


[9] Running LPM (OLS) models (robustness) ...
  LPM ANC 4+ Visits                   N=5,186  R2=0.1400
  LPM Facility Delivery               N=5,469  R2=0.1693
  LPM Doctor-Attended ANC             N=5,187  R2=0.0886
  LPM Iron Supplementation            N=5,185  R2=0.0689
  LPM 1st ANC in 1st Trimester        N=4,758  R2=0.1124


In [21]:
# ---------------------------------------------------------------------------
# SECTION 10 -- EXPORT TABLE 2 (probit AME, all outcomes)
# ---------------------------------------------------------------------------
print("\n[10] Exporting Table 2 (probit AME) ...")

t2_rows = []
for i, vname in enumerate(VAR_NAMES):
    if vname == "const":
        continue
    row = {"Variable": REG_LABELS.get(vname, vname)}
    for col, r in probit_results.items():
        stars = sig_stars(r["pval"][i])
        row[r["label"]]            = f"{r['ame'][i]:.4f}{stars}"
        row[r["label"] + " (SE)"]  = f"({r['ame_se'][i]:.4f})"
    t2_rows.append(row)

for stat_name, key, fmt in [("N", "n", ",d"), ("McFadden R2", "pseudo_r2", ".4f")]:
    row = {"Variable": stat_name}
    for col, r in probit_results.items():
        row[r["label"]]           = format(r[key], fmt)
        row[r["label"] + " (SE)"] = ""
    t2_rows.append(row)

table2_df = pd.DataFrame(t2_rows)
table2_df.to_excel(os.path.join(OUT_DIR, "table2_probit_ame.xlsx"), index=False)
table2_df.to_csv( os.path.join(OUT_DIR,  "table2_probit_ame.csv"),  index=False)
print("    Saved table2_probit_ame.xlsx / .csv")


[10] Exporting Table 2 (probit AME) ...
    Saved table2_probit_ame.xlsx / .csv


In [22]:
# ---------------------------------------------------------------------------
# SECTION 11 -- EXPORT TABLE 3 (probit AME vs LPM, ANC 4+ only)
# ---------------------------------------------------------------------------
print("\n[11] Exporting Table 3 (probit vs LPM for ANC 4+) ...")

t3_rows = []
for i, vname in enumerate(VAR_NAMES):
    if vname == "const":
        continue
    pr = probit_results["anc4"]
    lp = lpm_results["anc4"]
    t3_rows.append({
        "Variable":    REG_LABELS.get(vname, vname),
        "Probit AME":  f"{pr['ame'][i]:.4f}{sig_stars(pr['pval'][i])}",
        "Probit SE":   f"({pr['ame_se'][i]:.4f})",
        "LPM Coef.":   f"{lp['coef'][i]:.4f}{sig_stars(lp['pval'][i])}",
        "LPM SE":      f"({lp['se'][i]:.4f})",
    })
for stat_name in ["N", "R2 / Pseudo R2"]:
    pr = probit_results["anc4"]
    lp = lpm_results["anc4"]
    t3_rows.append({
        "Variable":    stat_name,
        "Probit AME":  f"{pr['n']:,}" if stat_name == "N" else f"{pr['pseudo_r2']:.4f}",
        "Probit SE":   "",
        "LPM Coef.":   f"{lp['n']:,}" if stat_name == "N" else f"{lp['r2']:.4f}",
        "LPM SE":      "",
    })

table3_df = pd.DataFrame(t3_rows)
table3_df.to_excel(os.path.join(OUT_DIR, "table3_probit_vs_lpm.xlsx"), index=False)
table3_df.to_csv( os.path.join(OUT_DIR,  "table3_probit_vs_lpm.csv"),  index=False)
print("    Saved table3_probit_vs_lpm.xlsx / .csv")


[11] Exporting Table 3 (probit vs LPM for ANC 4+) ...
    Saved table3_probit_vs_lpm.xlsx / .csv


In [24]:
# ---------------------------------------------------------------------------
# SECTION 12 -- FIGURE 3: MARGINAL EFFECTS BAR CHART WITH CI
# ---------------------------------------------------------------------------
print("\n[12] Generating Figure 3 (marginal effects plot) ...")

EDU_VARS   = ["edu_primary", "edu_secondary", "edu_higher"]
EDU_COLORS = ["#fc8d59", "#2166ac", "#1a9641"]
EDU_DISP   = ["Primary\n(ref: no edu)", "Secondary\n(ref: no edu)", "Higher\n(ref: no edu)"]

fig, ax = plt.subplots(figsize=(11, 6))
x       = np.arange(len(OUTCOMES))
width   = 0.22

for j, (evar, color, elbl) in enumerate(zip(EDU_VARS, EDU_COLORS, EDU_DISP)):
    idx    = VAR_NAMES.index(evar)
    mfx    = [probit_results[col]["ame"][idx]    for col in OUTCOMES]
    mfx_se = [probit_results[col]["ame_se"][idx] for col in OUTCOMES]
    pos    = x + (j - 1) * width
    ax.bar(pos, mfx, width=width, label=elbl,
           color=color, alpha=0.85, edgecolor="white")
    ax.errorbar(pos, mfx, yerr=1.96 * np.array(mfx_se),
                fmt="none", color="black", capsize=3, linewidth=1)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(list(OUTCOMES.values()), fontsize=9.5)
ax.set_ylabel("Average Marginal Effect (probability units)", fontsize=11)
ax.set_title(
    "Average Marginal Effects of Maternal Education on Health Service Demand\n"
    "(Probit Models with 95% CI bars, BDHS 2022)",
    fontweight="bold", fontsize=11
)
ax.legend(title="Education Level", fontsize=9.5, title_fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "fig3_mfx.png"), dpi=160, bbox_inches="tight")
plt.close()
print("    Saved fig3_mfx.png")


[12] Generating Figure 3 (marginal effects plot) ...
    Saved fig3_mfx.png


In [32]:
# ---------------------------------------------------------------------------
# SECTION 13 -- HETEROGENEITY: RURAL vs URBAN SUBSAMPLE
# ---------------------------------------------------------------------------
print("\n[13] Heterogeneity analysis (rural vs urban) ...")

REGRESSORS_NORURAL = [r for r in REGRESSORS if r != "rural"]
VAR_NAMES_NR       = ["const"] + REGRESSORS_NORURAL

# Use v025 directly from the original df for subsetting
# v025: 1=urban, 2=rural — no float/NaN issues
urban_ids = df.index[df["v025"] == 1]
rural_ids  = df.index[df["v025"] == 2]

subgroup_rows = []
for id_index, sg_label in [(rural_ids, "Rural"), (urban_ids, "Urban")]:
    df_sub = df_clean.loc[df_clean.index.isin(id_index)].copy()
    print(f"    Processing {sg_label}: {len(df_sub):,} rows")
    for col, lbl in OUTCOMES.items():
        sub = df_sub[[col] + REGRESSORS_NORURAL].dropna()
        if len(sub) < 50:
            continue
        y = sub[col].values
        X = sm.add_constant(sub[REGRESSORS_NORURAL].values)
        try:
            for method in ["newton", "bfgs", "nm"]:
                try:
                    res = Probit(y, X).fit(
                        disp=False, maxiter=300, method=method,
                        warn_convergence=False
                    )
                    _ = res.cov_params()
                    break
                except Exception:
                    continue

            phi_mean = norm.pdf(X @ res.params).mean()
            ame      = phi_mean * res.params
            ame_se   = phi_mean * np.sqrt(np.diag(res.cov_params()))
            p        = 2 * (1 - norm.cdf(np.abs(ame / ame_se)))
            for i, vname in enumerate(VAR_NAMES_NR):
                if vname in EDU_VARS:
                    subgroup_rows.append({
                        "Subgroup":  sg_label,
                        "Outcome":   lbl,
                        "Variable":  REG_LABELS.get(vname, vname),
                        "AME":       round(ame[i], 4),
                        "SE":        round(ame_se[i], 4),
                        "z":         round(ame[i] / ame_se[i], 3),
                        "p-value":   round(p[i], 4),
                        "Sig":       sig_stars(p[i]),
                        "N":         len(sub),
                    })
        except Exception as e:
            print(f"    Warning ({sg_label}, {lbl}): {e}")

subgroup_df = pd.DataFrame(subgroup_rows)
subgroup_df.to_excel(os.path.join(OUT_DIR, "table4_rural_urban.xlsx"), index=False)
print("    Saved table4_rural_urban.xlsx")
print("\n    Rural vs Urban -- ANC 4+ education effects:")
print(subgroup_df[subgroup_df["Outcome"] == "ANC 4+ Visits"].to_string(index=False))


[13] Heterogeneity analysis (rural vs urban) ...
    Processing Rural: 4,180 rows
    Processing Urban: 2,135 rows
    Saved table4_rural_urban.xlsx

    Rural vs Urban -- ANC 4+ education effects:
Subgroup       Outcome                Variable    AME     SE     z  p-value Sig    N
   Rural ANC 4+ Visits   Primary (ref: no edu) 0.1049 0.0417 2.513   0.0120  ** 3480
   Rural ANC 4+ Visits Secondary (ref: no edu) 0.1757 0.0412 4.269   0.0000 *** 3480
   Rural ANC 4+ Visits    Higher (ref: no edu) 0.2773 0.0451 6.150   0.0000 *** 3480
   Urban ANC 4+ Visits   Primary (ref: no edu) 0.0063 0.0550 0.114   0.9092     1706
   Urban ANC 4+ Visits Secondary (ref: no edu) 0.1599 0.0527 3.033   0.0024 *** 1706
   Urban ANC 4+ Visits    Higher (ref: no edu) 0.3169 0.0569 5.571   0.0000 *** 1706


In [33]:
# ---------------------------------------------------------------------------
# SECTION 14 -- FIGURE 4: RURAL vs URBAN COMPARISON (ANC 4+)
# ---------------------------------------------------------------------------
print("\n[14] Generating Figure 4 (rural vs urban) ...")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax_i, (sg_label, color_map) in enumerate([
        ("Urban", ["#fc8d59","#2166ac","#1a9641"]),
        ("Rural", ["#fc8d59","#2166ac","#1a9641"])]):
    ax  = axes[ax_i]
    sub = subgroup_df[
        (subgroup_df["Subgroup"] == sg_label) &
        (subgroup_df["Outcome"]  == "ANC 4+ Visits")
    ]
    ax.bar(sub["Variable"], sub["AME"], color=color_map,
           edgecolor="white", alpha=0.85)
    ax.errorbar(sub["Variable"], sub["AME"], yerr=1.96 * sub["SE"],
                fmt="none", color="black", capsize=4)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"{sg_label} Subsample", fontweight="bold", fontsize=11)
    ax.set_ylabel("Average Marginal Effect", fontsize=10)
    ax.set_xticklabels(sub["Variable"], rotation=12, ha="right", fontsize=8.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle(
    "Education Effects on ANC 4+ by Urban/Rural Subsample\n"
    "(Probit AME with 95% CI, BDHS 2022)",
    fontweight="bold", fontsize=11
)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "fig4_rural_urban.png"), dpi=160, bbox_inches="tight")
plt.close()
print("    Saved fig4_rural_urban.png")


[14] Generating Figure 4 (rural vs urban) ...
    Saved fig4_rural_urban.png


In [34]:
# ---------------------------------------------------------------------------
# SECTION 15 -- DIVISIONAL ANALYSIS & FIGURE 5
# ---------------------------------------------------------------------------
print("\n[15] Division-level analysis ...")

div_data = (
    df.groupby("v024")["anc4"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"v024":"div_code", "mean":"anc4_pct", "count":"n"})
)
div_data["division"]  = div_data["div_code"].map(DIV_LABELS)
div_data["anc4_pct"] *= 100
div_data = div_data.sort_values("anc4_pct").reset_index(drop=True)
div_data.to_excel(os.path.join(OUT_DIR, "tableC_by_division.xlsx"), index=False)

fig, ax = plt.subplots(figsize=(8, 5))
nat_mean = div_data["anc4_pct"].mean()
colors   = ["#d73027" if v < nat_mean else "#4575b4" for v in div_data["anc4_pct"]]
ax.barh(div_data["division"], div_data["anc4_pct"],
        color=colors, edgecolor="white")
ax.axvline(nat_mean, color="gray", linestyle="--", linewidth=1.3,
           label=f"National mean ({nat_mean:.1f}%)")
for _, row in div_data.iterrows():
    ax.text(row["anc4_pct"] + 0.5, row.name,
            f"{row['anc4_pct']:.1f}%", va="center", fontsize=9)
ax.set_xlabel("ANC 4+ Prevalence (%)", fontsize=11)
ax.set_title("ANC 4+ Utilisation by Division (BDHS 2022)",
             fontweight="bold", fontsize=11)
ax.legend(fontsize=9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "fig5_division.png"), dpi=160, bbox_inches="tight")
plt.close()
print("    Saved fig5_division.png  +  tableC_by_division.xlsx")


[15] Division-level analysis ...
    Saved fig5_division.png  +  tableC_by_division.xlsx


In [35]:
# ---------------------------------------------------------------------------
# SECTION 16 -- COMPILE ALL TABLES INTO ONE EXCEL WORKBOOK
# ---------------------------------------------------------------------------
print("\n[16] Compiling master Excel workbook (all_tables.xlsx) ...")

with pd.ExcelWriter(os.path.join(OUT_DIR, "all_tables.xlsx"), engine="openpyxl") as writer:
    desc_df.to_excel(     writer, sheet_name="Table1_Descriptives",  index=False)
    edu_table.to_excel(   writer, sheet_name="TableA_ByEducation",   index=False)
    wealth_table.to_excel(writer, sheet_name="TableB_ByWealth",      index=False)
    table2_df.to_excel(   writer, sheet_name="Table2_Probit_AME",    index=False)
    table3_df.to_excel(   writer, sheet_name="Table3_Probit_vs_LPM", index=False)
    subgroup_df.to_excel( writer, sheet_name="Table4_RuralUrban",    index=False)
    div_data.to_excel(    writer, sheet_name="TableC_ByDivision",    index=False)

print("    Saved all_tables.xlsx  (7 sheets)")


[16] Compiling master Excel workbook (all_tables.xlsx) ...
    Saved all_tables.xlsx  (7 sheets)


In [36]:
# ---------------------------------------------------------------------------
# SECTION 17 -- FINAL SUMMARY
# ---------------------------------------------------------------------------
print("\n" + "="*65)
print("  FINAL SUMMARY -- KEY EDUCATION MARGINAL EFFECTS (PROBIT AME)")
print("="*65)
print(f"  {'Outcome':<30} {'Primary':>10} {'Secondary':>12} {'Higher':>10}")
print(f"  {'-'*64}")

for col, r in probit_results.items():
    pi  = VAR_NAMES.index("edu_primary")
    si  = VAR_NAMES.index("edu_secondary")
    hi  = VAR_NAMES.index("edu_higher")
    print(
        f"  {r['label']:<30} "
        f"{r['ame'][pi]:>7.4f}{sig_stars(r['pval'][pi]):<3} "
        f"{r['ame'][si]:>7.4f}{sig_stars(r['pval'][si]):<3} "
        f"{r['ame'][hi]:>7.4f}{sig_stars(r['pval'][hi]):<3}"
    )

print("\n  *** p<0.01   ** p<0.05   * p<0.10")
print(f"\n  All outputs saved to: {OUT_DIR}")
print("="*65)
print("\n[DONE]")


  FINAL SUMMARY -- KEY EDUCATION MARGINAL EFFECTS (PROBIT AME)
  Outcome                           Primary    Secondary     Higher
  ----------------------------------------------------------------
  ANC 4+ Visits                   0.0740**   0.1720***  0.3006***
  Facility Delivery               0.0313     0.1283***  0.2906***
  Doctor-Attended ANC             0.0671***  0.1238***  0.1997***
  Iron Supplementation            0.0774***  0.1525***  0.2421***
  1st ANC in 1st Trimester        0.0148     0.0989***  0.2275***

  *** p<0.01   ** p<0.05   * p<0.10

  All outputs saved to: E:\7th semester Econometrics\data\output

[DONE]
